# Live Inference Test: Compare Base vs Fine-tuned (GGUF)

Use llama.cpp server to generate summaries for 10 test samples.
Compare base model vs fine-tuned adapter side-by-side.

In [1]:
import json
import sys
import requests
from pathlib import Path
from IPython.display import display, HTML
from tqdm.notebook import tqdm

# Add config to path
sys.path.insert(0, str(Path.cwd().parent / "config"))
from prompt_loader import get_inference_prompt_base_model

## Configuration

In [2]:
# llama.cpp server endpoint
MODEL_URL = "http://localhost:8100"  # Fine-tuned model server

# Test data
TEST_FILE = "../../eval/test_set.jsonl"  # Change if using different test set
NUM_SAMPLES = 10

# Generation config
GEN_CONFIG = {
    "max_tokens": 1024,
    "temperature": 0.7,
    "top_p": 0.9,
    "stream": False,
    "stop": ["\n\n\n", "###", "Government Report:", "Expert Summary:", "<|endoftext|>"],  # Stop sequences
}

## Check Server Status

In [3]:
def check_server(url: str, name: str) -> bool:
    """Check if llama.cpp server is running."""
    try:
        resp = requests.get(f"{url}/health", timeout=5)
        if resp.status_code == 200:
            print(f"✓ {name} server is running at {url}")
            return True
    except Exception as e:
        print(f"❌ {name} server not reachable at {url}")
        print(f"   Error: {e}")
    return False

# Check server
model_available = check_server(MODEL_URL, "Model")

if not model_available:
    print("\nTo start model server:")
    print(f"  llama-server --model ~/models/checkpoint-1600-gguf/qwen3-0.6b.Q4_K_M.gguf --port 8100 --ctx-size 32768 -ngl 99")

✓ Model server is running at http://localhost:8100


## Load Test Data

In [4]:
# Load test samples
test_samples = []
test_file = Path(TEST_FILE)

if test_file.exists():
    with open(test_file) as f:
        for i, line in enumerate(f):
            if i >= NUM_SAMPLES:
                break
            test_samples.append(json.loads(line))
    print(f"✓ Loaded {len(test_samples)} test samples")
else:
    print(f"❌ Test file not found: {test_file}")
    print("Please run: cd ../../eval && python create_test_set.py")

✓ Loaded 10 test samples


## Generate Summaries

In [5]:
def generate_summary(server_url: str, document: str, config: dict) -> str:
    """Generate summary using llama.cpp server."""
    prompt = get_inference_prompt_base_model(document)
    
    payload = {
        "prompt": prompt,
        "n_predict": config["max_tokens"],
        "temperature": config["temperature"],
        "top_p": config["top_p"],
        "stream": config["stream"],
        "stop": config.get("stop", []),  # Add stop sequences
    }
    
    try:
        resp = requests.post(
            f"{server_url}/completion",
            json=payload,
            timeout=120,
        )
        resp.raise_for_status()
        
        result = resp.json()
        summary = result.get("content", "").strip()
        return summary
        
    except Exception as e:
        print(f"Error generating summary: {e}")
        return f"[ERROR: {e}]"

In [6]:
# Generate summaries for all test samples
results = []

for sample in tqdm(test_samples, desc="Generating summaries"):
    doc = sample['document']
    ref = sample['summary']
    sample_id = sample.get('sample_id', 'unknown')
    category = sample.get('category', 'unknown')
    
    # Generate with model
    model_summary = None
    if model_available:
        model_summary = generate_summary(MODEL_URL, doc, GEN_CONFIG)
    
    results.append({
        'sample_id': sample_id,
        'category': category,
        'document': doc,
        'reference': ref,
        'model_summary': model_summary,
    })

print(f"\n✓ Generated summaries for {len(results)} samples")

Generating summaries:   0%|          | 0/10 [00:00<?, ?it/s]


✓ Generated summaries for 10 samples


## Display Results

In [7]:
def display_comparison(result):
    """Display side-by-side comparison of summaries."""
    doc = result['document']
    ref = result['reference']
    model_summary = result['model_summary']
    
    model_html = ""
    if model_summary:
        model_html = f"""
        <div style="margin: 15px 0;">
            <h4 style="background-color: #cfe2ff; padding: 8px;">🎯 Model Output ({len(model_summary.split())} words)</h4>
            <pre style="white-space: pre-wrap; background-color: white; padding: 10px; border: 1px solid #0d6efd;">{model_summary}</pre>
        </div>
    """
    
    html = f"""
    <div style="border: 2px solid #333; padding: 15px; margin: 20px 0; background-color: #f9f9f9;">
        <h3>Sample {result['sample_id']} - Category: {result['category']}</h3>
        
        <hr>
        
        <div style="margin: 15px 0;">
            <h4 style="background-color: #e0e0e0; padding: 8px;">📄 Document ({len(doc)} chars)</h4>
            <pre style="white-space: pre-wrap; background-color: white; padding: 10px; border: 1px solid #ccc; max-height: 300px; overflow-y: auto;">{doc}</pre>
        </div>
        
        <div style="margin: 15px 0;">
            <h4 style="background-color: #d4edda; padding: 8px;">✅ Reference Summary ({len(ref.split())} words)</h4>
            <pre style="white-space: pre-wrap; background-color: white; padding: 10px; border: 1px solid #28a745;">{ref}</pre>
        </div>
        
        {model_html}
    </div>
    """
    
    display(HTML(html))

In [8]:
# Display all results
for result in results:
    display_comparison(result)

## Quick Stats

In [ ]:
# Average word counts
if results:
    avg_ref_words = sum(len(r['reference'].split()) for r in results) / len(results)

    print("Average Summary Length:")
    print(f"  Reference:  {avg_ref_words:.1f} words")

    model_summaries = [r['model_summary'] for r in results if r['model_summary']]
    if model_summaries:
        avg_model_words = sum(len(s.split()) for s in model_summaries) / len(model_summaries)
        print(f"  Model:      {avg_model_words:.1f} words")
else:
    print("No results to display stats")

## Save Results (Optional)

In [ ]:
# Save to JSONL for further analysis
output_file = Path("inference_test_gguf_results.jsonl")

with open(output_file, 'w') as f:
    for result in results:
        f.write(json.dumps(result) + '\n')

print(f"✓ Saved results to: {output_file}")

## Server Management Commands

### Start Model Server
```bash
llama-server \
  --model ~/models/checkpoint-1600-gguf/qwen3-0.6b.Q4_K_M.gguf \
  --port 8100 \
  --ctx-size 32768 \
  -ngl 99
```

### Check Server Health
```bash
curl http://localhost:8100/health
```